In [ ]:
import src.database.scripts.sql as sql 

import requests
from datetime import datetime, timedelta, timezone
import numpy as np

In [2]:
conn = sql.connect_pc()
cursor = conn.cursor()

query = """
SELECT name, rarity, vendor_price FROM item_data
WHERE type NOT IN ('Armor', 'Weapon', 'Accessory')
AND vendor_price > 100
ORDER BY vendor_price DESC
"""

cursor.execute(query)
fetch = cursor.fetchall()

In [ ]:
from_date = (datetime.now(timezone.utc) - timedelta(minutes=15)).strftime("%Y-%m-%dT%H:%M:%SZ")
with requests.Session() as ses:
    price_list = [('name', 'net', 'vendor', 'market', 'q_under')]
    for name, rarity, vendor_price in fetch:
        print(name, end='\r')
        url = f"https://api.darkerdb.com/v1/market?item={name.replace('\'', "’")}&rarity={rarity}&from={from_date}&limit=50"
        endpoint = ses.get(url)
        price = tuple(listing['price_per_unit'] for listing in endpoint.json()['body'])
        if not price: price = [vendor_price]       
        market_min = round(np.nanmin(price))
        q_under = len(list(filter(lambda x: x < vendor_price, price)))
        price_list.append((name, vendor_price - market_min, vendor_price, market_min, q_under))

In [4]:
price_list[1:] = sorted(price_list[1:], key=lambda x: x[1], reverse=True)
for x in price_list:
    print(f"{x[0]:<35} {x[1]:>10} {x[2]:>10} {x[3]:>10} {x[4]:>10}")

name                                       net     vendor     market    q_under
Rubysilver Ore                              75        125         50         22
Froststone Ore                              25        125        100         10
Froststone Powder                           25        125        100          1
Chronicles of the Cursed Crown              10        500        490          3
Obsidian Ore                                10        125        115          6
Ruby (Ultimate)                              5        175        170          1
Gold Crown (Ultimate)                        1        700        699          2
Corrupted Heart                              0      30000      30000          0
Head of the Midnight Stag                    0      20000      20000          0
Crown of the Midnight Stag                   0      10000      10000          0
Ancient Stingray Egg                         0      10000      10000          0
Frost Wyvern Egg                        